

Import dataset from hugging face



In [7]:
!pip -q install -U datasets huggingface_hub transformers pyarrow tqdm

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
#Configure project directories
import os

PROJECT_DIR = "/content/drive/MyDrive/SLM_Biomed"

TRAIN_DIR = os.path.join(PROJECT_DIR, "train")
TEST_DIR = os.path.join(PROJECT_DIR, "test")
LOG_DIR = os.path.join(PROJECT_DIR, "logs")
META_DIR = os.path.join(PROJECT_DIR, "metadata")

for directory in [
    PROJECT_DIR,
    TRAIN_DIR,
    TEST_DIR,
    LOG_DIR,
    META_DIR
]:
    os.makedirs(directory, exist_ok=True)

print("Project directory:")
print(PROJECT_DIR)

Project directory:
/content/drive/MyDrive/SLM_Biomed


In [10]:
#Keep Hugging Face cache in Colab
HF_CACHE = "/content/hf_cache"

os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE, "hub")
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE, "datasets")

os.makedirs(HF_CACHE, exist_ok=True)

print("Hugging Face cache:")
print(HF_CACHE)

Hugging Face cache:
/content/hf_cache


In [11]:
#Load BioMedLM tokenizer
from transformers import AutoTokenizer

TOKENIZER_NAME = "stanford-crfm/BioMedLM"

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME
)

print("Tokenizer:", TOKENIZER_NAME)
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/267 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/602k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/276k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Tokenizer: stanford-crfm/BioMedLM
Vocabulary size: 28895


Load biomed- enriched data in streaming mode

In [12]:
from datasets import load_dataset

DATASET_NAME = "almanach/Biomed-Enriched"

dataset = load_dataset(
    DATASET_NAME,
    split="commercial",
    streaming=True
)

print(dataset)

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

IterableDataset({
    features: ['id', 'article_id', 'path', 'text', 'language', 'section_title', 'domain', 'document_type', 'educational_score', 'domain_scores', 'document_type_scores', 'language_score', 'authors', 'article_url', 'license_url'],
    num_shards: 26
})


Quality FIlter: Inspect record

```
# This is formatted as code
```



In [13]:
MIN_EDUCATIONAL_SCORE = 3.5
MIN_LANGUAGE_SCORE = 0.95
MIN_BIOMEDICAL_SCORE = 0.90


def get_biomedical_score(row):
    scores = row.get("domain_scores")

    if scores is None:
        return 0.0

    return float(scores[0])


def quality_filter(row):

    text = row.get("text", "")

    if not text:
        return False

    if len(text.strip()) < 200:
        return False

    if row.get("language") != "en":
        return False

    if float(row.get("language_score", 0)) < MIN_LANGUAGE_SCORE:
        return False

    if row.get("domain") != "biomedical":
        return False

    if get_biomedical_score(row) < MIN_BIOMEDICAL_SCORE:
        return False

    if float(row.get("educational_score", 0)) < MIN_EDUCATIONAL_SCORE:
        return False

    return True

Article level test/Traqin Split

In [14]:
import hashlib


def article_split(article_id, test_fraction=0.03):

    digest = hashlib.sha256(
        article_id.encode("utf-8")
    ).hexdigest()

    value = int(digest[:12], 16) / float(16**12 - 1)

    if value < test_fraction:
        return "test"

    return "train"

# 10-20M token test

In [15]:
SMALL_TRAIN_TOKENS = 10_000_000
SMALL_TEST_TOKENS = 2_000_000

Small Test: Extraction Function:

In [21]:
import json
import time
from tqdm.auto import tqdm


def extract_corpus(
    train_target_tokens,
    test_target_tokens,
    train_output,
    test_output,
    test_fraction=0.03
):

    dataset = load_dataset(
        DATASET_NAME,
        split="commercial",
        streaming=True
    )

    train_tokens = 0
    test_tokens = 0

    train_records = 0
    test_records = 0

    train_articles = set()
    test_articles = set()

    start_time = time.time()

    with open(train_output, "w", encoding="utf-8") as train_file, \
         open(test_output, "w", encoding="utf-8") as test_file:

        for row in tqdm(dataset, desc="Streaming"):

            if not quality_filter(row):
                continue

            article_id = row.get("article_id")

            if not article_id:
                continue

            split = article_split(
                article_id,
                test_fraction
            )

            # BioMedLM tokenizer
            token_count = len(
                tokenizer.encode(
                    row["text"],
                    add_special_tokens=False, truncation=False
                )
            )

            if token_count == 0:
                continue

            record = {
                "article_id": article_id,
                "text": row["text"],
                "token_count": token_count,
                "section_title": row.get("section_title"),
                "document_type": row.get("document_type"),
                "educational_score": row.get("educational_score"),
                "article_url": row.get("article_url"),
                "license_url": row.get("license_url")
            }

            if (
                split == "train"
                and train_tokens < train_target_tokens
            ):

                train_file.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ) + "\n"
                )

                train_tokens += token_count
                train_records += 1
                train_articles.add(article_id)

            elif (
                split == "test"
                and test_tokens < test_target_tokens
            ):

                test_file.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ) + "\n"
                )

                test_tokens += token_count
                test_records += 1
                test_articles.add(article_id)

            if (
                train_tokens >= train_target_tokens
                and
                test_tokens >= test_target_tokens
            ):
                break

    elapsed = time.time() - start_time

    return {
        "train_tokens": train_tokens,
        "test_tokens": test_tokens,
        "train_records": train_records,
        "test_records": test_records,
        "train_articles": len(train_articles),
        "test_articles": len(test_articles),
        "elapsed_hours": elapsed / 3600,
        "tokenizer": TOKENIZER_NAME
    }

In [22]:
#Small Test

small_train_file = os.path.join(
    TRAIN_DIR,
    "train_10M.jsonl"
)

small_test_file = os.path.join(
    TEST_DIR,
    "test_2M.jsonl"
)

stats = extract_corpus(
    train_target_tokens=SMALL_TRAIN_TOKENS,
    test_target_tokens=SMALL_TEST_TOKENS,
    train_output=small_train_file,
    test_output=small_test_file
)

print(json.dumps(stats, indent=2))

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Streaming: 0it [00:00, ?it/s]

{
  "train_tokens": 10000004,
  "test_tokens": 2000001,
  "train_records": 45230,
  "test_records": 9077,
  "train_articles": 1888,
  "test_articles": 353,
  "elapsed_hours": 0.06781065596474542,
  "tokenizer": "stanford-crfm/BioMedLM"
}


In [23]:
#Check REsulting FIle Size:
def file_size_gb(path):
    return os.path.getsize(path) / (1024**3)


print(
    "Train size:",
    file_size_gb(small_train_file),
    "GB"
)

print(
    "Test size:",
    file_size_gb(small_test_file),
    "GB"
)

Train size: 0.05512753501534462 GB
Test size: 0.01114988699555397 GB


#Data Audit

In [24]:
from collections import Counter
from tqdm.auto import tqdm

AUDIT_RECORDS = 1_000_000

dataset = load_dataset(
    DATASET_NAME,
    split="commercial",
    streaming=True
)

total = 0
passed = 0

articles = set()
texts = set()

duplicate_texts = 0

doc_types = Counter()

total_tokens = 0

for row in tqdm(dataset, total=AUDIT_RECORDS):

    if total >= AUDIT_RECORDS:
        break

    total += 1

    if not quality_filter(row):
        continue

    passed += 1

    article_id = row.get("article_id")

    if article_id:
        articles.add(article_id)

    text = row["text"]

    # Exact text duplicate check
    if text in texts:
        duplicate_texts += 1
    else:
        texts.add(text)

    doc_types[row.get("document_type")] += 1

    total_tokens += len(
        tokenizer(
            text,
            add_special_tokens=False,
            truncation=False
        )["input_ids"]
    )


print("\n==============================")
print("1M RECORD AUDIT")
print("==============================")

print("Records examined :", total)
print("Passed filter    :", passed)
print("Unique articles  :", len(articles))
print("Duplicate texts  :", duplicate_texts)

print(
    "Pass percentage  :",
    round(100 * passed / total, 2)
)

print(
    "Tokens examined  :",
    f"{total_tokens:,}"
)

print("\nDocument types:")
for k, v in doc_types.most_common():
    print(k, v)

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

  0%|          | 0/1000000 [00:00<?, ?it/s]


1M RECORD AUDIT
Records examined : 1000000
Passed filter    : 626150
Unique articles  : 25035
Duplicate texts  : 91
Pass percentage  : 62.62
Tokens examined  : 136,722,226

Document types:
study 552650
review 57084
other 16070
clinical_case 346


make the final corpus

In [19]:
FINAL_TRAIN_TOKENS = 5_000_000_000
FINAL_TEST_TOKENS = 150_000_000

In [20]:
'''
final_train_file = os.path.join(
    TRAIN_DIR,
    "biomed_train_5B.jsonl"
)

final_test_file = os.path.join(
    TEST_DIR,
    "biomed_test_150M.jsonl"
)

final_stats = extract_corpus(
    train_target_tokens=FINAL_TRAIN_TOKENS,
    test_target_tokens=FINAL_TEST_TOKENS,
    train_output=final_train_file,
    test_output=final_test_file
)

print(json.dumps(final_stats, indent=2)) '''

'\nfinal_train_file = os.path.join(\n    TRAIN_DIR,\n    "biomed_train_5B.jsonl"\n)\n\nfinal_test_file = os.path.join(\n    TEST_DIR,\n    "biomed_test_150M.jsonl"\n)\n\nfinal_stats = extract_corpus(\n    train_target_tokens=FINAL_TRAIN_TOKENS,\n    test_target_tokens=FINAL_TEST_TOKENS,\n    train_output=final_train_file,\n    test_output=final_test_file\n)\n\nprint(json.dumps(final_stats, indent=2)) '